# 02 — Structural Metrics: CRR, WSSI, DI (ODB Pipeline)

Formulas (LaTeX):
$$ \mathrm{CRR} = \frac{N_{\text{cognate}}}{N_{\text{total}}} \times 100 $$

$$ \mathrm{WSSI} = \sum_i w_i \, s_i, \quad \sum_i w_i = 1,\; s_i \in [0,1] $$

$$ \mathrm{DI} = 100 - \mathrm{WSSI} $$

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

cog = pd.read_excel('/mnt/data/odb_raw_data_tables_readable.xlsx', 'Table_4_2_Cognates')
phon = pd.read_excel('/mnt/data/odb_raw_data_tables_readable.xlsx', 'Table_4_3_Phonology')
sem = pd.read_excel('/mnt/data/odb_raw_data_tables_readable.xlsx', 'Table_4_4_Semantic')
N = 207
shared = int(cog.loc[cog['Cognate type'].isin(['COG-PHON','IDENT','IDENT/COG']),'Count'].sum())
print('Shared cognate/identical items:', shared)

## 1. Cognate Retention Rate (CRR)

In [ ]:
CRR = shared / N * 100
print(f'CRR = {shared}/{N} × 100 = {CRR:.2f}%')

## 2. Weighted Structural Similarity Index (WSSI)

In [ ]:
# Similarity sub-scores s_i in [0,1], expert-weight expert-weight_i (sum = 1)
weights = {'lexical': 0.40, 'phonological': 0.30, 'semantic': 0.20, 'morphological': 0.10}

s_lex = shared / N                      # share of lexically shared items
s_phon = 1 - min(phon['Count'].sum(), N) / N   # fewer systematic shifts => more similarity
no_sem = int(sem.loc[sem['Semantic change']=='none','Count'].iloc[0])
s_sem = no_sem / N
s_morph = 0.70                          # morphological divergence score (expert estimate, inflectional parity)

WSSI = sum(weights[k]*v for k,v in zip(weights, [s_lex, s_phon, s_sem, s_morph]))
for k, s, w in zip(weights, [s_lex, s_phon, s_sem, s_morph], weights.values()):
    print(f'{k:14s} w={w:.2f}  s={s:.3f}  contrib={w*s:.3f}')
print(f'WSSI = {WSSI*100:.2f} / 100')

## 3. Divergence Index (DI)

In [ ]:
DI = 100 - WSSI*100
print(f'DI = 100 − WSSI = {DI:.2f}')
print(f'Consistency check: CRR ({CRR:.1f}%) > replacement rate ({100-CRR:.1f}%) → retention-dominant divergence.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))
axes[0].bar(['CRR','WSSI','DI'], [CRR, WSSI*100, DI], color=['#4C72B0','#55A868','#C44E52'])
for i, v in enumerate([CRR, WSSI*100, DI]): axes[0].text(i, v+1, f'{v:.1f}', ha='center')
axes[0].set_ylim(0, 105); axes[0].set_title('Structural metrics (Az–Tr)')

ks = list(weights); vs = [weights[k]*s for k,s in zip(weights,[s_lex,s_phon,s_sem,s_morph])]
axes[1].barh(ks, vs, color='#8172B3'); axes[1].set_title('WSSI contributions')
for i,v in enumerate(vs): axes[1].text(v+0.005, i, f'{v:.3f}', va='center')
plt.tight_layout(); plt.show()

## Interpretation
High CRR (~83%) with heavy phonological layer → the two varieties share a lexicon but diverge phonetically; WSSI ≈ 70/100 yields DI ≈ 30, a *moderate* structural divergence driven by sound change rather than replacement.